# RQ1: Emotional Tone (Sentiment Analysis)

**Research Question 1 - How does emotional tone differ between platforms and stances?**

This notebook scores every Reddit and YouTube comment with two complementary
lexicon/rule based methods, compares sentiment across platforms and political
stances, validates the differences with statistical tests, and exports a
sentiment-enriched dataset for the downstream modules (03 Topics, 04 Echo
Chambers, 05 Toxicity).

- **VADER** - valence-aware, social-media tuned compound score in [-1, 1].
- **TextBlob** - polarity in [-1, 1] and subjectivity in [0, 1].

**Stance labels:** `P` = Pro-Palestine, `I` = Pro-Israel, `N` = Neutral.

## 1. Setup and Imports

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from scipy import stats
from tqdm.auto import tqdm
import warnings

warnings.filterwarnings("ignore")
tqdm.pandas()

plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 10

STANCE_ORDER = ["P", "I", "N"]
STANCE_NAMES = {"P": "Pro-Palestine", "I": "Pro-Israel", "N": "Neutral"}
STANCE_COLORS = {"P": "#2ecc71", "I": "#3498db", "N": "#95a5a6"}
SENT_COLORS = {"positive": "#2ecc71", "neutral": "#95a5a6", "negative": "#e74c3c"}


def find_repo_root():
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "data" / "reddit_labeled.csv").exists():
            return candidate
    return here


REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "02_emotional_tone_analysis" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Libraries imported successfully")
print(f"Data dir  : {DATA_DIR}")
print(f"Output dir: {OUTPUT_DIR}")

## 2. Load Labeled Data

In [ ]:
print("Loading labeled data...")
reddit_df = pd.read_csv(DATA_DIR / "reddit_labeled.csv")
youtube_df = pd.read_csv(DATA_DIR / "youtube_labeled.csv")

# Keep only the three valid stance labels
reddit_df = reddit_df[reddit_df["Label"].isin(STANCE_ORDER)].copy()
youtube_df = youtube_df[youtube_df["Label"].isin(STANCE_ORDER)].copy()

# Parse timestamps for the temporal section
reddit_df["created_time"] = pd.to_datetime(reddit_df["created_time"], errors="coerce")
youtube_df["created_time"] = pd.to_datetime(youtube_df["created_time"], errors="coerce")

# Text columns differ per platform
REDDIT_TEXT, YOUTUBE_TEXT = "self_text", "text"

print(f"Reddit : {len(reddit_df):,} comments")
print(f"YouTube: {len(youtube_df):,} comments")

## 3. Sentiment Scoring Functions

VADER returns a `compound` score plus positive/neutral/negative proportions.
We label a comment positive if compound >= 0.05, negative if <= -0.05, else
neutral (the standard VADER thresholds). TextBlob adds polarity and a
subjectivity score.

In [ ]:
vader = SentimentIntensityAnalyzer()


def vader_scores(text):
    if not isinstance(text, str) or text == "":
        return (0.0, 0.0, 0.0, 0.0, "neutral")
    s = vader.polarity_scores(text)
    if s["compound"] >= 0.05:
        label = "positive"
    elif s["compound"] <= -0.05:
        label = "negative"
    else:
        label = "neutral"
    return (s["compound"], s["pos"], s["neu"], s["neg"], label)


def textblob_scores(text):
    if not isinstance(text, str) or text == "":
        return (0.0, 0.0, "neutral")
    blob = TextBlob(text)
    pol = blob.sentiment.polarity
    subj = blob.sentiment.subjectivity
    if pol > 0.1:
        label = "positive"
    elif pol < -0.1:
        label = "negative"
    else:
        label = "neutral"
    return (pol, subj, label)


VADER_COLS = ["vader_compound", "vader_pos", "vader_neu", "vader_neg", "vader_label"]
TB_COLS = ["textblob_polarity", "textblob_subjectivity", "textblob_label"]
print("Sentiment scoring functions defined")

## 4. Apply Sentiment Scoring

Scoring runs over the full dataset (~1.5M comments). A single pass per method is
used (progress bars shown); expect a few minutes total.

In [ ]:
def score_frame(df, text_col):
    texts = df[text_col].fillna("").astype(str)
    vad = pd.DataFrame(texts.progress_map(vader_scores).tolist(),
                       columns=VADER_COLS, index=df.index)
    tb = pd.DataFrame(texts.progress_map(textblob_scores).tolist(),
                      columns=TB_COLS, index=df.index)
    return pd.concat([df, vad, tb], axis=1)


print("Scoring Reddit...")
reddit_df = score_frame(reddit_df, REDDIT_TEXT)
print("Scoring YouTube...")
youtube_df = score_frame(youtube_df, YOUTUBE_TEXT)

print("\nReddit VADER label distribution:")
print(reddit_df["vader_label"].value_counts())
print("\nYouTube VADER label distribution:")
print(youtube_df["vader_label"].value_counts())

## 5. Sentiment Distribution Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
order = ["positive", "neutral", "negative"]

panels = [
    (axes[0, 0], reddit_df, "vader_label", "Reddit Sentiment (VADER)"),
    (axes[0, 1], youtube_df, "vader_label", "YouTube Sentiment (VADER)"),
    (axes[1, 0], reddit_df, "textblob_label", "Reddit Sentiment (TextBlob)"),
    (axes[1, 1], youtube_df, "textblob_label", "YouTube Sentiment (TextBlob)"),
]
for ax, df, col, title in panels:
    counts = df[col].value_counts().reindex(order).fillna(0)
    bars = ax.bar(order, counts.values, color=[SENT_COLORS[o] for o in order],
                  alpha=0.85, edgecolor="black")
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_ylabel("Count")
    ax.grid(axis="y", alpha=0.3)
    total = counts.sum()
    for b, v in zip(bars, counts.values):
        ax.text(b.get_x() + b.get_width() / 2, v, f"{v:,.0f}\n({v/total*100:.1f}%)",
                ha="center", va="bottom", fontsize=9, fontweight="bold")
plt.tight_layout()
plt.show()

## 6. Sentiment by Stance

Mean VADER compound score and sentiment-label mix for each stance, per platform.

In [ ]:
def stance_sentiment_table(df, name):
    print("=" * 70)
    print(f"{name}: VADER sentiment statistics by stance")
    print("=" * 70)
    tbl = df.groupby("Label").agg(
        n=("vader_compound", "size"),
        compound_mean=("vader_compound", "mean"),
        compound_median=("vader_compound", "median"),
        compound_std=("vader_compound", "std"),
        pos=("vader_pos", "mean"),
        neg=("vader_neg", "mean"),
    ).reindex(STANCE_ORDER).round(3)
    tbl.index = [STANCE_NAMES[s] for s in tbl.index]
    print(tbl)
    return tbl


_ = stance_sentiment_table(reddit_df, "REDDIT")
_ = stance_sentiment_table(youtube_df, "YOUTUBE")

In [ ]:
# Heatmaps: sentiment mix (%) by stance
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, df, title in [(axes[0], reddit_df, "Reddit"), (axes[1], youtube_df, "YouTube")]:
    ct = pd.crosstab(df["Label"], df["vader_label"], normalize="index") * 100
    ct = ct.reindex(index=STANCE_ORDER, columns=["positive", "neutral", "negative"])
    ct.index = [STANCE_NAMES[s] for s in ct.index]
    sns.heatmap(ct, annot=True, fmt=".1f", cmap="RdYlGn", ax=ax,
                cbar_kws={"label": "% of stance"})
    ax.set_title(f"{title}: Sentiment by Stance (%)", fontsize=12, fontweight="bold")
    ax.set_xlabel("Sentiment"); ax.set_ylabel("Stance")
plt.tight_layout()
plt.show()

In [ ]:
# Box plots: compound score by stance
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, df, title in [(axes[0], reddit_df, "Reddit"), (axes[1], youtube_df, "YouTube")]:
    sns.boxplot(data=df, x="Label", y="vader_compound", order=STANCE_ORDER,
                hue="Label", hue_order=STANCE_ORDER, palette=STANCE_COLORS,
                legend=False, showfliers=False, ax=ax)
    ax.set_xticklabels([STANCE_NAMES[s] for s in STANCE_ORDER])
    ax.axhline(0, color="red", linestyle="--", alpha=0.5)
    ax.set_title(f"{title}: VADER Compound by Stance", fontsize=12, fontweight="bold")
    ax.set_xlabel("Stance"); ax.set_ylabel("Compound score")
plt.tight_layout()
plt.show()

## 7. Polarity vs Subjectivity

TextBlob polarity against subjectivity. Plotted on a stratified sample per stance
(full data would be ~1.5M overlapping points).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
SAMPLE_PER_STANCE = 4000
for ax, df, title in [(axes[0], reddit_df, "Reddit"), (axes[1], youtube_df, "YouTube")]:
    for stance in STANCE_ORDER:
        sub = df[df["Label"] == stance]
        if len(sub) > SAMPLE_PER_STANCE:
            sub = sub.sample(SAMPLE_PER_STANCE, random_state=42)
        ax.scatter(sub["textblob_subjectivity"], sub["textblob_polarity"],
                   alpha=0.3, s=12, c=STANCE_COLORS[stance], label=STANCE_NAMES[stance])
    ax.axhline(0, color="black", linestyle="--", alpha=0.3)
    ax.axvline(0.5, color="black", linestyle="--", alpha=0.3)
    ax.set_title(f"{title}: Polarity vs Subjectivity", fontsize=12, fontweight="bold")
    ax.set_xlabel("Subjectivity"); ax.set_ylabel("Polarity")
    ax.legend()
plt.tight_layout()
plt.show()

## 8. Platform Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
order = ["positive", "neutral", "negative"]
for ax, col, title in [(axes[0], "vader_label", "VADER"),
                       (axes[1], "textblob_label", "TextBlob")]:
    comp = pd.DataFrame({
        "Reddit": reddit_df[col].value_counts(normalize=True) * 100,
        "YouTube": youtube_df[col].value_counts(normalize=True) * 100,
    }).reindex(order).fillna(0)
    comp.plot(kind="bar", ax=ax, color=["#3498db", "#e74c3c"], alpha=0.85,
              edgecolor="black", width=0.8)
    ax.set_title(f"Sentiment Comparison ({title})", fontsize=12, fontweight="bold")
    ax.set_xlabel("Sentiment"); ax.set_ylabel("Percentage (%)")
    ax.legend(title="Platform")
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=0)
    for container in ax.containers:
        ax.bar_label(container, fmt="%.1f%%", padding=2)
plt.tight_layout()
plt.show()

## 9. Statistical Significance Tests

We validate the visual differences with formal tests:

- **Chi-square** - is the sentiment-label mix independent of stance? (per platform)
- **Kruskal-Wallis** - do compound-score distributions differ across the three stances?
- **Mann-Whitney U** - does the compound-score distribution differ between platforms?

Cramer's V is reported as an effect size for the chi-square tests because, with
~1.5M observations, even trivial differences reach significance.

In [ ]:
def cramers_v(confusion):
    chi2 = stats.chi2_contingency(confusion)[0]
    n = confusion.to_numpy().sum()
    r, k = confusion.shape
    return np.sqrt((chi2 / n) / (min(r - 1, k - 1)))


print("CHI-SQUARE: stance vs sentiment label")
for name, df in [("Reddit", reddit_df), ("YouTube", youtube_df)]:
    ct = pd.crosstab(df["Label"], df["vader_label"])
    chi2, p, dof, _ = stats.chi2_contingency(ct)
    print(f"  {name}: chi2={chi2:,.1f}, dof={dof}, p={p:.3e}, "
          f"Cramer's V={cramers_v(ct):.3f}")

print("\nKRUSKAL-WALLIS: VADER compound across stances")
for name, df in [("Reddit", reddit_df), ("YouTube", youtube_df)]:
    groups = [df[df["Label"] == s]["vader_compound"].dropna() for s in STANCE_ORDER]
    h, p = stats.kruskal(*groups)
    print(f"  {name}: H={h:,.1f}, p={p:.3e}")

print("\nMANN-WHITNEY U: Reddit vs YouTube compound")
u, p = stats.mannwhitneyu(reddit_df["vader_compound"].dropna(),
                          youtube_df["vader_compound"].dropna(),
                          alternative="two-sided")
print(f"  U={u:,.0f}, p={p:.3e}")
print(f"  Reddit mean compound = {reddit_df['vader_compound'].mean():.4f}")
print(f"  YouTube mean compound = {youtube_df['vader_compound'].mean():.4f}")

## 10. Temporal Sentiment Trends

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
fig.suptitle("Monthly Mean Sentiment (VADER Compound)", fontsize=14, fontweight="bold")
for ax, df, title, color in [(axes[0], reddit_df, "Reddit", "#3498db"),
                             (axes[1], youtube_df, "YouTube", "#e74c3c")]:
    ts = (df.dropna(subset=["created_time"])
            .groupby(df["created_time"].dt.to_period("M"))["vader_compound"].mean())
    ts.index = ts.index.astype(str)
    ax.plot(ts.index, ts.values, marker="o", color=color, linewidth=2)
    ax.axhline(0, color="k", linestyle="--", alpha=0.3)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_ylabel("Mean compound")
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 11. Export Sentiment-Enriched Data

Saved to `02_emotional_tone_analysis/outputs/`. Modules 03-05 load these files
(and fall back to recomputing VADER from `data/` if they are absent).

In [ ]:
reddit_out = OUTPUT_DIR / "reddit_with_sentiment.csv"
youtube_out = OUTPUT_DIR / "youtube_with_sentiment.csv"

reddit_df.to_csv(reddit_out, index=False, encoding="utf-8")
youtube_df.to_csv(youtube_out, index=False, encoding="utf-8")

print(f"Exported: {reddit_out}  ({reddit_df.shape[0]:,} x {reddit_df.shape[1]})")
print(f"Exported: {youtube_out} ({youtube_df.shape[0]:,} x {youtube_df.shape[1]})")
print("\nNew sentiment columns:", VADER_COLS + TB_COLS)

## Summary - RQ1 Findings

- **Method agreement** - VADER and TextBlob are compared side by side; VADER
  (social-media tuned) is the primary score used downstream.
- **Platform contrast** - the platform comparison and Mann-Whitney test quantify
  how Reddit and YouTube differ in overall emotional tone.
- **Stance effect** - the by-stance tables, heatmaps and Kruskal-Wallis tests show
  how Pro-Palestine, Pro-Israel and Neutral comments differ in sentiment within
  each platform, with chi-square + Cramer's V giving effect sizes.
- **Temporal** - monthly mean sentiment traces how tone shifts over the collection window.

The sentiment-enriched datasets are exported for the Topics (03), Echo Chambers
(04) and Toxicity (05) modules.